![image.png](https://i.imgur.com/a3uAqnb.png)

# RAG System Implementation - Homework Assignment

In this homework, you will implement a **Retrieval-Augmented Generation (RAG) system** for question answering. This project will help you understand the fundamentals of document retrieval, embedding systems, and how to combine retrieval with language models for improved performance.

## 📌 Project Overview
- **Task**: Build a RAG system to answer questions based on retrieved context
- **Dataset**: RAG Dataset 12000 from neural-bridge
- **Baseline Performance**: 33.00% accuracy with basic chunking and small LLM
- **Goal**: Either replicate the baseline OR achieve better performance (aim for >30%)

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand document chunking and retrieval strategies
- Learn about embedding models and vector similarity search
- Implement FAISS vector stores for efficient retrieval
- Practice combining retrieval with language generation
- Explore optimization techniques for RAG systems
- Understand evaluation challenges in generative AI systems

## ⚠️ Important Constraints
- **No model fine-tuning allowed** - use pre-trained models only
- **Maximum model size**: 300M parameters for the generation model
- **Goal**: Either achieve baseline performance (30.00%) OR exceed it
- **Note**: You can choose to implement the exact baseline OR create your own improved version

## 🎯 Evaluation Methodology Note

**Important**: The accuracy calculation in this assignment uses **cosine similarity between generated and ground truth answers** with a 0.5 threshold. This method has several limitations:

1. **Semantic vs. Exact Matching**: Cosine similarity captures semantic meaning but may miss exact factual correctness
2. **Threshold Sensitivity**: The 0.5 threshold is arbitrary - some correct answers might score below it
3. **Length Bias**: Very short answers might have inflated similarity scores
4. **Paraphrasing Issues**: Correct paraphrases might score lower than expected
5. **False Positives**: Semantically similar but factually incorrect answers might score high

**Why We Use This Method**: Despite limitations, cosine similarity provides a reasonable approximation for semantic correctness at scale, which is more practical than manual evaluation of 2,400+ question-answer pairs.

**Real-World Note**: Production RAG systems typically use multiple evaluation metrics including exact match, BLEU scores, human evaluation, and domain-specific correctness checks.

In [1]:
# !pip install faiss-cpu
# !pip install -U langchain-community
# !pip install bitsandbytes

## 1️⃣ Initial Setup and Environment

**Task**: Set up the required libraries and check GPU availability.

**Requirements**:
- Import all necessary libraries for RAG implementation
- Check CUDA availability for GPU acceleration
- Set up proper logging and warning suppression

In [2]:
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    pipeline
)
from datasets import load_dataset
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
import numpy as np
from tqdm import tqdm
import gc
import warnings
import os
import logging
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Suppress warnings and logs
warnings.filterwarnings('ignore')
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2️⃣ Dataset Loading and Exploration

**Task**: Load the RAG dataset and explore its structure.

**Requirements**:
- Load the neural-bridge/rag-dataset-12000 dataset
- Convert to pandas DataFrame for easier manipulation
- Analyze the dataset structure and create a large document for retrieval
- Display basic statistics about the dataset

In [ ]:
print("Loading RAG Dataset 12000...")
dataset = load_dataset("neural-bridge/rag-dataset-12000", split="test")

df = pd.DataFrame(dataset)
print(f"Dataset loaded with {len(df)} examples")

all_contexts = df['context'].unique()
large_document = "\n\n".join(all_contexts)
print(f"Combined document created with {len(large_document):,} characters")

Loading RAG Dataset 12000...


README.md: 0.00B [00:00, ?B/s]

(…)-00000-of-00001-9df3a936e1f63191.parquet:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

(…)-00000-of-00001-af2a9f454ad1b8a3.parquet:   0%|          | 0.00/5.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Dataset loaded with 2400 examples
Combined document created with 8,250,984 characters


## 3️⃣ Text Chunking and Vector Store Setup

**Task**: Implement document chunking and create a vector store for retrieval.

**Options**:
- **Option A**: Use baseline parameters (chunk_size=124, overlap=10)
- **Option B**: Experiment with your own chunking strategy

**Requirements**:
- Choose and implement a text chunking strategy
- Set up embedding model for vectorization
- Create FAISS vector store from document chunks

In [ ]:
# TODO: Choose your chunking strategy
# Option A: Baseline approach
# TODO: Create RecursiveCharacterTextSplitter with chunk_size=124, chunk_overlap=10

# Option B: Your custom approach
# TODO: Create RecursiveCharacterTextSplitter with your own parameters
# TODO: Experiment with different chunk sizes and overlap values

# TODO: Set up text chunking and embeddings
print("Setting up text chunking and embeddings...")

# TODO: Split the large document into chunks using your text_splitter
# TODO: Convert chunks to Document objects

# TODO: Choose and initialize your embedding model
# TODO: Consider different sentence transformer models

# TODO: Create FAISS vector store from documents and embeddings
print("Vector store created successfully!")
# TODO: Print the number of chunks created

Setting up text chunking and embeddings...
Vector store created successfully!
Number of chunks: 87283


## 4️⃣ Language Model Selection and Setup

**Task**: Choose and set up a small language model for answer generation.

**Options**:
- **Option A**: Use baseline model (google/flan-t5-base)
- **Option B**: Choose your own model (<300M parameters)

**Baseline Configuration**:
- Model: google/flan-t5-base (~250M parameters)
- Temperature: 0.3
- Max length: 100 tokens

In [ ]:
# TODO: Choose your language model approach

# Option A: Baseline model
# TODO: Set model name to "google/flan-t5-base"
# TODO: Load tokenizer using AutoTokenizer.from_pretrained()
# TODO: Load model using AutoModelForSeq2SeqLM.from_pretrained() with torch_dtype=torch.float16
# TODO: Create text2text-generation pipeline with appropriate parameters

# Option B: Your custom model
# TODO: Choose a different small model (<300M parameters)
# TODO: Load your chosen tokenizer and model
# TODO: Set up your model pipeline with your preferred parameters

print("Small LLM pipeline created!")

Loading small LLM for RAG system...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Small LLM pipeline created!


## 5️⃣ Evaluation Framework Setup

**Task**: Set up the evaluation framework using cosine similarity-based assessment.

**Evaluation Method Explanation**:
Our evaluation uses cosine similarity between generated answers and ground truth with a 0.5 threshold. This approach:
- ✅ Captures semantic similarity between answers
- ✅ Works at scale without manual evaluation
- ❌ May miss exact factual correctness
- ❌ Threshold (0.5) is somewhat arbitrary
- ❌ Can give false positives for semantically similar but wrong answers

**Requirements**:
- Load sentence transformer for answer comparison
- Implement similarity-based evaluation function

In [ ]:
# TODO: Load sentence transformer model for evaluation
# TODO: Use SentenceTransformer to load 'all-MiniLM-L6-v2' or another suitable model
print("Model loaded!")

# TODO: Implement evaluate_answer_correctness function
def evaluate_answer_correctness(generated_answer, ground_truth, threshold=0.5):
   """
   Evaluate answer correctness using cosine similarity.

   Limitations:
   - May not capture exact factual accuracy
   - Threshold is somewhat arbitrary
   - Short answers may have inflated scores
   - Paraphrases might score lower than expected
   """
   # TODO: Clean and normalize both answers (strip whitespace, convert to lowercase)
   
   # TODO: Encode both answers using the sentence model
   
   # TODO: Calculate cosine similarity between the embeddings
   
   # TODO: Determine if answer is correct based on threshold
   
   # TODO: Return both correctness boolean and similarity score
   pass

Loading sentence transformer model for evaluation...
Model loaded!


## 6️⃣ RAG Implementation

**Task**: Implement your RAG question-answering system.

**Baseline Approach** (if replicating):
- Retrieve top-2 most similar chunks
- Concatenate chunks as context
- Use prompt: "Answer the question based on the context. Context: {context} Question: {question}"
- Generate with temperature=0.3, max_length=50

**Your Approach** (if improving):
- Experiment with different retrieval strategies (k value, similarity thresholds)
- Try different prompt engineering approaches
- Consider context processing improvements
- Optimize generation parameters

In [ ]:
# TODO: Implement your RAG function

def rag_answer_question(question, vectorstore, k=2):
   """
   Generate an answer using RAG approach.

   Args:
       question: Input question
       vectorstore: FAISS vector store for retrieval
       k: Number of chunks to retrieve

   Returns:
       Generated answer string
   """
   # TODO: Use vectorstore.similarity_search to retrieve top k relevant documents
   
   # TODO: Extract page_content from retrieved documents and combine into context
   
   # TODO: Engineer your prompt template (experiment with different formats)
   
   # TODO: Use your LLM pipeline to generate answer
   # TODO: Experiment with max_length, temperature, and other generation parameters
   
   # TODO: Extract and return the generated text
   pass

# TODO: Test your RAG system
# TODO: Get a test question from the dataset
# TODO: Generate an answer using your RAG function
# TODO: Print the question, generated answer, and ground truth for comparison

Test Question: Who is the music director of the Quebec Symphony Orchestra?
Generated Answer: Fabien Gabel
Ground Truth: The music director of the Quebec Symphony Orchestra is Fabien Gabel.


## 7️⃣ Comprehensive Evaluation

**Task**: Evaluate your RAG system on the complete dataset.

**Baseline Performance to Match/Beat**:
- **Total Questions**: 2,399
- **Correct Answers**: 720  
- **Overall Accuracy**: 30.00%
- **Average Similarity**: 0.380
  - NOTE: some of the questions and answers were null, so be careful!!!
  - NOTE: The baseline might be a bit different for you as an LLM is not deterministic
    
**Requirements**:
- Run evaluation on the full dataset
- Track both accuracy and average similarity
- Handle errors gracefully
- Display progress and final results

In [ ]:
# TODO: Implement comprehensive evaluation function
def run_evaluation(dataset_df, vectorstore, num_samples=None, similarity_threshold=0.5):
   """
   Run comprehensive evaluation of RAG system.

   Note: This evaluation uses cosine similarity which has limitations
   for measuring true correctness, but provides a reasonable approximation.
   """
   # TODO: Handle sampling - use full dataset if num_samples is None, otherwise sample
   
   # TODO: Initialize tracking variables (results list, correct_count, total_similarity)
   
   # TODO: Print evaluation start message with sample count
   
   # TODO: Loop through each row in the sample dataset with progress bar
   
       # TODO: Extract question and ground truth answer from row
       
       # TODO: Skip if ground_truth is None or NaN
       
       # TODO: Generate answer using your rag_answer_question function
       
       # TODO: Skip if generated_answer is None
       
       # TODO: Evaluate answer correctness using your evaluate_answer_correctness function
       
       # TODO: Update correct_count if answer is correct
       
       # TODO: Add similarity to total_similarity
       
       # TODO: Append result dictionary to results list
       
       # TODO: Handle exceptions gracefully and continue evaluation
   
   # TODO: Calculate final accuracy and average similarity
   
   # TODO: Return results, final_accuracy, and avg_similarity

# TODO: Run your evaluation on the dataset
# TODO: Print starting message
# TODO: Call run_evaluation with your vectorstore
# TODO: Store the returned results, accuracy, and avg_similarity

Starting RAG evaluation...
Evaluating on 2400 samples...


100%|███████████████████████████████████████████████████████████████████████████████| 2400/2400 [08:14<00:00,  4.86it/s]


In [11]:

print(f"\nFINAL RESULTS")
print(f"{'='*50}")
print(f"Total Questions Evaluated: {len(evaluation_results)}")
print(f"Correct Answers: {sum(r['correct'] for r in evaluation_results)}")
print(f"Overall Accuracy: {accuracy:.2%}")
print(f"Average Similarity Score: {avg_similarity:.3f}")
print(f"{'='*50}")


FINAL RESULTS
Total Questions Evaluated: 2399
Correct Answers: 720
Overall Accuracy: 30.01%
Average Similarity Score: 0.382


## 📏 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Quality (40%)**
  - Correct RAG system implementation
  - Proper use of embedding models and vector stores
  - Valid model selection within constraints (<300M params)
  - Code quality and documentation

2. **Performance & Analysis (35%)**
  - Achieving reasonable performance (aim for >30% accuracy)
  - Thoughtful approach to optimization (if attempted)
  - Understanding of system components and their trade-offs

3. **Code Quality & Documentation (25%)**
  - Clean, readable, and well-structured code
  - Proper error handling and edge case management
  - Clear variable naming and function documentation
  - Efficient implementation and resource management